# 31 (MLA) — Examples Of PySpark ML

**ML Analyst perspective.** A faithful adaptation of the classic PySpark linear-regression-on-salary example, run entirely against IRIS through IrisPark. `age` + `Experience` assemble an `Independent Features` vector that predicts `Salary`; a linear model is fit on a train split and evaluated on a held-out test split.

In [1]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

Connected to IRIS: <irispark.session.IrisParkSession object at 0xffff9c17eb00>


In [2]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Read the dataset

The 6-row salary dataset lives at `notebooks/test1.csv` (same data as the original example).

In [3]:
import os

# Kernel runs from the notebook's directory; also accept repo-root relative.
candidates = ["test1.csv", os.path.join("..", "notebooks", "test1.csv")]
csv_path = next(p for p in candidates if os.path.exists(p))
training = session.read.csv(csv_path)
training.show()

        Name  age  Experience  Salary
0      Krish   31          10   30000
1  Sudhanshu   30           8   25000
2      Sunny   29           4   20000
3       Paul   24           3   20000
4     Harsha   21           1   15000
5    Shubham   23           2   18000
6       Ravi   26           5   21000
7      Priya   32          12   34000
8      Anita   28           6   23000
9     Vikram   35          15   41000


In [4]:
training.printSchema()

root
 |-- Name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- Experience: integer (nullable = true)
 |-- Salary: integer (nullable = true)


In [5]:
training.columns

['Name', 'age', 'Experience', 'Salary']

## 2. Feature engineering

`[age, Experience]` become the independent features feeding the regressor. (The original example names the column `Independent Features`; IrisPark requires identifier-safe column names, so we use `Independent_Features`.)

In [6]:
from irispark.ml.feature import VectorAssembler

featureassembler = VectorAssembler(inputCols=["age", "Experience"], outputCol="Independent_Features")
output = featureassembler.transform(training)
output.show()

        Name  age  Experience  Salary Independent_Features
0      Krish   31          10   30000                31,10
1  Sudhanshu   30           8   25000                 30,8
2      Sunny   29           4   20000                 29,4
3       Paul   24           3   20000                 24,3
4     Harsha   21           1   15000                 21,1
5    Shubham   23           2   18000                 23,2
6       Ravi   26           5   21000                 26,5
7      Priya   32          12   34000                32,12
8      Anita   28           6   23000                 28,6
9     Vikram   35          15   41000                35,15


/home/user/irispark-mvp/irispark/ml/base.py:49: UserWarning: VectorAssembler emits a comma-joined string column, not a numeric vector (IRIS has no native vector type). ML estimators consume the raw featuresCol columns directly.
  return self._transform(df)


In [7]:
output.columns

['Name', 'age', 'Experience', 'Salary', 'Independent_Features']

## 3. Finalize the modeling frame

Keep the numeric features and the label (`Salary`). The `Independent_Features` vector column is illustrative — IrisPark's `LinearRegression.fit` consumes the raw numeric feature columns directly.

In [8]:
finalized_data = output.select("age", "Experience", "Salary")
finalized_data.show()

   age  Experience  Salary
0   31          10   30000
1   30           8   25000
2   29           4   20000
3   24           3   20000
4   21           1   15000
5   23           2   18000
6   26           5   21000
7   32          12   34000
8   28           6   23000
9   35          15   41000


## 4. Train / test split

`randomSplit` returns two frames; we materialize each into a temp table so the later `transform` (which wraps the frame in a subquery) doesn't trip over the split's `%ID`-based filter.

In [9]:
train_data, test_data = finalized_data.randomSplit([0.75, 0.25], seed=1)
train_data.write.mode('overwrite').saveAsTable('pyspark_ml_train')
test_data.write.mode('overwrite').saveAsTable('pyspark_ml_test')
train_data = session.table('pyspark_ml_train')
test_data = session.table('pyspark_ml_test')
print('train rows:', train_data.count(), '| test rows:', test_data.count())

train rows: 15 | test rows: 5


## 5. Fit Linear Regression

IrisPark's `LinearRegression` fits closed-form normal equations (ridge-regularized) with numpy.

In [10]:
from irispark.ml.regression import LinearRegression

regressor = LinearRegression(featuresCol=["age", "Experience"], labelCol="Salary")
regressor = regressor.fit(train_data)

### Coefficients

In [11]:
regressor.coefficients

[-182.11246762853347, 1838.6052534220985]

### Intercept

In [12]:
regressor.intercept

17604.051054383413

## 6. Predictions & metrics

PySpark's `regressor.evaluate(test).predictions` and `.meanAbsoluteError` / `.meanSquaredError` are now supported directly — the `evaluate()` shim delegates to `transform` + `RegressionEvaluator` under the hood.

In [13]:
pred_results = regressor.evaluate(test_data)
pred_results.predictions.select("age", "Experience", "Salary", "prediction").show()
print((pred_results.meanAbsoluteError, pred_results.meanSquaredError))

   age  Experience  Salary            prediction
0   27           4   20000  20041.43544210140331
1   38          18   47000  43778.67184609691414
2   29           7   25000  25193.02626711063187
3   36          14   39000  36788.47576766558708
4   34          13   36000  35314.09544950055552
(1270.6437291897957, 3155447.1186767253)


## API mapping vs PySpark

- `Model.evaluate(test).predictions` → supported by `evaluate()`; returns a summary whose `.predictions` is the scored DataFrame.
- `evaluateResult.meanAbsoluteError` / `meanSquaredError` → summary properties, computed via `RegressionEvaluator`.
- Classification models expose `evaluate()` returning a `BinaryClassificationSummary` with `.accuracy` / `.precision` / `.recall` / `.fMeasureByLabel` / `.areaUnderROC`.
- `VectorAssembler` here emits the assembled features as a comma-joined string column (`"31.0,10.0"`), not a numeric vector — the fit still uses the raw `featuresCol` columns directly, so results match.
- Column names must be identifier-safe (no spaces), hence `Independent_Features` instead of `Independent Features`.

In [14]:
session.sql('DROP TABLE IF EXISTS pyspark_ml_train')
session.sql('DROP TABLE IF EXISTS pyspark_ml_test')

SQLResult(rows=0, columns=[])

In [15]:
if session is not None:
    session.close()
    print("Session closed.")

Session closed.
